# DeepLense Test 1 - Multi-Class Lens Classification

Three-class classifier for `no`, `sphere`, and `vort`.

Evaluation protocol:
- split `dataset/train` into a stratified 90:10 `train` / internal `val`
- use internal `val` for model selection and primary reported metrics
- use `dataset/val` only once at the end as a supplemental holdout evaluation


In [ ]:
import copy
import random
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    classification_report,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import label_binarize
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from torchvision.transforms import InterpolationMode

plt.style.use("seaborn-v0_8-whitegrid")

In [ ]:
# Config
SEED = 42
VAL_SIZE = 0.1
BATCH_SIZE = 256
EPOCHS = 50
LR = 3e-4
WEIGHT_DECAY = 1e-4
LABEL_SMOOTHING = 0.05
EARLY_STOP_PATIENCE = 5
NUM_WORKERS = 0

# Paths
cwd = Path.cwd()
ROOT = cwd if (cwd / "dataset").exists() else cwd / "test1"
DATA_ROOT = ROOT / "dataset"
ARTIFACTS = ROOT / "artifacts"
ARTIFACTS.mkdir(parents=True, exist_ok=True)

CLASS_NAMES = ["no", "sphere", "vort"]
CLASS_LABELS = ["No substructure", "Subhalo substructure", "Vortex substructure"]

# Seed
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
pin_memory = device.type == "cuda"

print(f"Device: {device}")
print(f"Data root: {DATA_ROOT}")

In [ ]:
# Load data


def load_split(split_name):
    paths, labels = [], []
    for label, class_name in enumerate(CLASS_NAMES):
        class_dir = DATA_ROOT / split_name / class_name
        class_paths = sorted(class_dir.glob("*.npy"))
        if not class_paths:
            raise FileNotFoundError(f"No .npy files found in {class_dir}")
        paths.extend(class_paths)
        labels.extend([label] * len(class_paths))
    return paths, labels


full_train_paths, full_train_labels = load_split("train")
holdout_paths, holdout_labels = load_split("val")

train_paths, val_paths, train_labels, val_labels = train_test_split(
    full_train_paths,
    full_train_labels,
    test_size=VAL_SIZE,
    stratify=full_train_labels,
    random_state=SEED,
)

split_counts = pd.DataFrame(
    {
        "split": ["train"] * len(train_labels)
        + ["internal_val"] * len(val_labels)
        + ["holdout"] * len(holdout_labels),
        "label": train_labels + val_labels + holdout_labels,
    }
)
split_counts["class_name"] = split_counts["label"].map(lambda i: CLASS_NAMES[i])
display(split_counts.groupby(["split", "class_name"]).size().unstack(fill_value=0))

print(f"Train samples: {len(train_paths):,}")
print(f"Internal val samples: {len(val_paths):,}")
print(f"Holdout samples: {len(holdout_paths):,}")

In [ ]:
# Compute normalization stats from the current training split only

total_sum = 0.0
total_sq = 0.0
total_pixels = 0

for path in train_paths:
    array = np.load(path).astype(np.float32)
    total_sum += float(array.sum())
    total_sq += float((array**2).sum())
    total_pixels += array.size

mean = total_sum / total_pixels
std = np.sqrt(max(total_sq / total_pixels - mean**2, 1e-12))

print(f"Mean: {mean:.6f}")
print(f"Std: {std:.6f}")

In [ ]:
# Dataset and DataLoaders


class LensDataset(Dataset):
    def __init__(self, paths, labels, transform=None):
        self.paths = paths
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, index):
        image = torch.from_numpy(np.load(self.paths[index]).astype(np.float32))
        if self.transform is not None:
            image = self.transform(image)
        return image, self.labels[index]


train_transform = transforms.Compose(
    [
        transforms.RandomHorizontalFlip(),
        transforms.RandomVerticalFlip(),
        transforms.RandomRotation(20, interpolation=InterpolationMode.BILINEAR),
        transforms.Normalize([mean], [std]),
    ]
)

eval_transform = transforms.Normalize([mean], [std])

train_loader = DataLoader(
    LensDataset(train_paths, train_labels, train_transform),
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=pin_memory,
)
val_loader = DataLoader(
    LensDataset(val_paths, val_labels, eval_transform),
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=pin_memory,
)
holdout_loader = DataLoader(
    LensDataset(holdout_paths, holdout_labels, eval_transform),
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=pin_memory,
)

sample_images, sample_labels = next(iter(train_loader))
print(f"Batch image shape: {tuple(sample_images.shape)}")
print(f"Batch labels shape: {tuple(sample_labels.shape)}")

In [ ]:
# Model


class ResidualBlock(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, 3, stride, 1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.conv2 = nn.Conv2d(out_channels, out_channels, 3, 1, 1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_channels)
        self.shortcut = (
            nn.Sequential(
                nn.Conv2d(in_channels, out_channels, 1, stride, bias=False),
                nn.BatchNorm2d(out_channels),
            )
            if stride != 1 or in_channels != out_channels
            else nn.Identity()
        )

    def forward(self, x):
        out = torch.nn.functional.silu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        return torch.nn.functional.silu(out + self.shortcut(x))


class LensResNet(nn.Module):
    def __init__(self, num_classes=3):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv2d(1, 32, 5, 2, 2, bias=False),
            nn.BatchNorm2d(32),
            nn.SiLU(),
        )
        self.layer1 = nn.Sequential(ResidualBlock(32, 32), ResidualBlock(32, 32))
        self.layer2 = nn.Sequential(ResidualBlock(32, 64, 2), ResidualBlock(64, 64))
        self.layer3 = nn.Sequential(ResidualBlock(64, 128, 2), ResidualBlock(128, 128))
        self.layer4 = nn.Sequential(ResidualBlock(128, 256, 2), ResidualBlock(256, 256))
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.head = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(0.3),
            nn.Linear(256, num_classes),
        )

    def forward(self, x):
        x = self.stem(x)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        x = self.pool(x)
        return self.head(x)


model = LensResNet(num_classes=len(CLASS_NAMES)).to(device)
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
# Training helpers


def evaluate(model, loader, criterion):
    model.eval()
    total_loss = 0.0
    all_labels, all_probs = [], []

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device, non_blocking=pin_memory)
            labels = labels.to(device, non_blocking=pin_memory)
            logits = model(images)
            loss = criterion(logits, labels)
            probs = torch.softmax(logits, dim=1)

            total_loss += loss.item() * len(images)
            all_labels.append(labels.cpu())
            all_probs.append(probs.cpu())

    labels_np = torch.cat(all_labels).numpy()
    probs_np = torch.cat(all_probs).numpy()
    preds_np = probs_np.argmax(axis=1)

    return {
        "loss": total_loss / len(loader.dataset),
        "accuracy": float((preds_np == labels_np).mean()),
        "auc_macro": float(
            roc_auc_score(labels_np, probs_np, multi_class="ovr", average="macro")
        ),
        "labels": labels_np,
        "predictions": preds_np,
        "probabilities": probs_np,
    }


criterion = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)
optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="max", factor=0.5, patience=2
)

best_auc = -float("inf")
best_state = copy.deepcopy(model.state_dict())
patience_counter = 0
history = []

for epoch in range(1, EPOCHS + 1):
    model.train()
    train_loss = 0.0
    train_correct = 0
    train_total = 0

    for images, labels in train_loader:
        images = images.to(device, non_blocking=pin_memory)
        labels = labels.to(device, non_blocking=pin_memory)

        optimizer.zero_grad(set_to_none=True)
        logits = model(images)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()

        train_loss += loss.item() * len(images)
        train_correct += (logits.argmax(dim=1) == labels).sum().item()
        train_total += len(images)

    val_metrics = evaluate(model, val_loader, criterion)
    scheduler.step(val_metrics["auc_macro"])
    current_lr = optimizer.param_groups[0]["lr"]

    history.append(
        {
            "epoch": epoch,
            "train_loss": train_loss / train_total,
            "train_acc": train_correct / train_total,
            "val_loss": val_metrics["loss"],
            "val_acc": val_metrics["accuracy"],
            "val_auc": val_metrics["auc_macro"],
            "lr": current_lr,
        }
    )

    print(
        f"Epoch {epoch:02d} | train_loss={train_loss/train_total:.4f} train_acc={train_correct/train_total:.4f} | "
        f"val_loss={val_metrics['loss']:.4f} val_acc={val_metrics['accuracy']:.4f} val_auc={val_metrics['auc_macro']:.4f} | "
        f"lr={current_lr:.2e}"
    )

    if val_metrics["auc_macro"] > best_auc:
        best_auc = val_metrics["auc_macro"]
        best_state = copy.deepcopy(model.state_dict())
        torch.save(
            {
                "model": best_state,
                "history": history,
                "best_val_auc": best_auc,
                "seed": SEED,
            },
            ARTIFACTS / "best_model.pt",
        )
        patience_counter = 0
    else:
        patience_counter += 1
        if patience_counter >= EARLY_STOP_PATIENCE:
            print("Early stopping")
            break

model.load_state_dict(best_state)
print(f"Best internal val AUC: {best_auc:.4f}")

In [ ]:
# Plot training curves

df = pd.DataFrame(history)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))
ax1.plot(df["epoch"], df["train_loss"], label="train")
ax1.plot(df["epoch"], df["val_loss"], label="internal val")
ax1.set_xlabel("Epoch")
ax1.set_ylabel("Loss")
ax1.legend()

ax2.plot(df["epoch"], df["train_acc"], label="train acc")
ax2.plot(df["epoch"], df["val_acc"], label="internal val acc")
ax2.plot(df["epoch"], df["val_auc"], label="internal val auc")
ax2.set_xlabel("Epoch")
ax2.legend()

plt.tight_layout()
plt.show()

In [ ]:
# Primary evaluation on internal validation split

val_results = evaluate(model, val_loader, criterion)
val_true = val_results["labels"]
val_pred = val_results["predictions"]
val_prob = val_results["probabilities"]
val_true_bin = label_binarize(val_true, classes=[0, 1, 2])

print("=" * 60)
print("INTERNAL VALIDATION RESULTS")
print("=" * 60)
print(f"Loss: {val_results['loss']:.4f}")
print(f"Accuracy: {val_results['accuracy']:.4f}")
print(f"Macro AUC: {val_results['auc_macro']:.4f}")

print("Per-class AUC:")
for i, class_name in enumerate(CLASS_NAMES):
    class_auc = roc_auc_score(val_true_bin[:, i], val_prob[:, i])
    print(f"  {class_name}: {class_auc:.4f}")

print("Classification Report:")
print(classification_report(val_true, val_pred, target_names=CLASS_NAMES))


In [ ]:
# Supplemental evaluation on holdout split

holdout_results = evaluate(model, holdout_loader, criterion)
y_true = holdout_results["labels"]
y_pred = holdout_results["predictions"]
y_prob = holdout_results["probabilities"]
y_true_bin = label_binarize(y_true, classes=[0, 1, 2])

print("=" * 60)
print("HOLDOUT RESULTS (SUPPLEMENTAL)")
print("=" * 60)
print(f"Loss: {holdout_results['loss']:.4f}")
print(f"Accuracy: {holdout_results['accuracy']:.4f}")
print(f"Macro AUC: {holdout_results['auc_macro']:.4f}")

print("Per-class AUC:")
for i, class_name in enumerate(CLASS_NAMES):
    class_auc = roc_auc_score(y_true_bin[:, i], y_prob[:, i])
    print(f"  {class_name}: {class_auc:.4f}")

print("Classification Report:")
print(classification_report(y_true, y_pred, target_names=CLASS_NAMES))


In [ ]:
# ROC curves and confusion matrices for both evaluation splits


def plot_results(title, labels, predictions, probabilities):
    labels_bin = label_binarize(labels, classes=[0, 1, 2])
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

    for i, class_name in enumerate(CLASS_NAMES):
        fpr, tpr, _ = roc_curve(labels_bin[:, i], probabilities[:, i])
        class_auc = roc_auc_score(labels_bin[:, i], probabilities[:, i])
        ax1.plot(fpr, tpr, label=f"{class_name} (AUC={class_auc:.3f})")

    ax1.plot([0, 1], [0, 1], "k--")
    ax1.set_xlabel("False Positive Rate")
    ax1.set_ylabel("True Positive Rate")
    ax1.set_title(f"{title} ROC Curves")
    ax1.legend()

    ConfusionMatrixDisplay.from_predictions(
        labels,
        predictions,
        display_labels=CLASS_NAMES,
        ax=ax2,
        cmap="Blues",
        colorbar=False,
    )
    ax2.set_title(f"{title} Confusion Matrix")

    plt.tight_layout()
    plt.show()


plot_results("Internal Validation", val_true, val_pred, val_prob)
plot_results("Holdout", y_true, y_pred, y_prob)